In [0]:
from pyspark.sql.functions import *

df = spark.table("hive_metastore.bronze.refunds")
df = df.withColumn("refund_date", date_format(col("refund_timestamp"), "yyyy-MM-dd")) \
    .withColumn("refund_time", date_format(col("refund_timestamp"), "HH:mm:ss")) \
        .drop("refund_timestamp")

df= df.withColumn("refund_reason_split", split(col("refund_reason"), ":")) \
    .withColumn("refund_type", col("refund_reason_split").getItem(0)) \
        .withColumn("refund_source", col("refund_reason_split").getItem(1))\
            .drop("refund_reason_split")


display(df)

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS hive_metastore.silver;

DROP table IF EXISTS hive_metastore.silver.refunds;

CREATE TABLE IF NOT EXISTS hive_metastore.silver.refunds
AS
select refund_id,
 payment_id,
cast(date_format(refund_timestamp, "yyyy-MM-dd") as date) as refund_date,
 date_format(refund_timestamp, "HH:mm:ss") as refund_time,
 refund_amount,
 split(refund_reason, ":")[0] as refund_type,
 split(refund_reason, ":")[1] as refund_source
 from hive_metastore.bronze.refunds;

In [0]:
%sql
DESC EXTENDED hive_metastore.silver.refunds